# EDA: Feature Distributions Analysis

This notebook visualizes the distributions of numeric and categorical variables in the kidney disease dataset.
Figures are saved in grayscale for publication quality.

## 1. Load and Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for grayscale output
plt.style.use('grayscale')
sns.set_palette('gray')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Note: savefig dpi will be set per figure based on publication guidelines:
# - Black-and-white images: ~600 dpi (histograms)
# - Grayscale images: 150-300 dpi (bar charts, pie charts)

# Load interim dataset
interim_path = Path("../data/interim/kidney_disease_interim.csv")
df = pd.read_csv(interim_path)

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

## 2. Visualize Numeric Variable Distributions

In [ ]:
# Create histograms for numeric variables
n_numeric = len(numeric_cols)
n_cols = 3
n_rows = (n_numeric + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    
    # Create histogram (black-and-white)
    ax.hist(df[col], bins=20, alpha=0.6, color='black', edgecolor='black', linewidth=0.5)
    ax2 = ax.twinx()
    ax2.set_ylabel('Density')
    ax2.grid(False)
    
    ax.set_title(f'{col}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

# Remove extra subplots
for idx in range(n_numeric, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()

# Save figure
# Using 600 dpi for black-and-white images as per publication guidelines
output_dir = Path("../reports/paper/figures")
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / "numeric_distributions_histograms.png", bbox_inches='tight', dpi=600)
print("Saved: numeric_distributions_histograms.png (600 dpi - black-and-white)")
plt.show()

## 3. Visualize Categorical Variable Distributions

For categorical variables, we create bar charts showing the count/frequency of each category.

In [ ]:
# Create bar charts for categorical variables
n_cat = len(categorical_cols)
n_cols = 3
n_rows = (n_cat + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    ax = axes[idx]
    
    # Count values and sort
    value_counts = df[col].value_counts().sort_values(ascending=False)
    
    # Create bar plot (grayscale)
    bars = ax.bar(range(len(value_counts)), value_counts.values, color='gray', edgecolor='black', linewidth=1)
    
    # Set x-axis labels with better formatting
    labels = [str(v)[:15] for v in value_counts.index]  # Truncate long labels
    ax.set_xticks(range(len(value_counts)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    
    ax.set_title(f'{col}', fontweight='bold', fontsize=11)
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=8)

# Remove extra subplots
for idx in range(n_cat, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()

# Save figure
# Using 300 dpi for grayscale images as per publication guidelines
fig.savefig(output_dir / "categorical_distributions_barcharts.png", bbox_inches='tight', dpi=300)
print("Saved: categorical_distributions_barcharts.png (300 dpi - grayscale)")
plt.show()

## 4. Visualize Target Variable Distribution

In [ ]:
# Pie chart for target (class) distribution only
target_col = 'class'
class_counts = df[target_col].value_counts()

fig_target, ax_target = plt.subplots(figsize=(6, 6))
ax_target.pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=['gray', 'lightgray'],
    wedgeprops={'edgecolor': 'black', 'linewidth': 1},
    textprops={'fontsize': 10}
)

ax_target.set_title("Target (class) Distribution", fontweight='bold')
ax_target.axis('equal')

# Using 300 dpi for grayscale images as per publication guidelines
fig_target.savefig(output_dir / "target_class_distribution_pie.png", bbox_inches='tight', dpi=300)
print("Saved: target_class_distribution_pie.png (300 dpi - grayscale)")
plt.show()